# ARAPPAV — End-to-End Math Verification Training

This notebook downloads the **Hendrycks MATH** dataset and runs the full
ARAPPAV self-play pipeline in **math mode**:

1. **Perturber** injects misconception-based errors into math solutions
2. **Verifier** tries to detect and explain those errors
3. Both models are trained adversarially via GRPO + DPO

## Requirements
- At least 1 GPU with 24+ GB VRAM (2 GPUs recommended)
- `pip install -e ".[dev]"` completed
- Hugging Face token set (optional, for higher rate limits)

## 1. Setup and Imports

In [ ]:
import json
import logging
import random
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from datasets import Dataset, DatasetDict
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid")

# Ensure package is importable
sys.path.insert(0, str(Path.cwd().parent / "src"))

# ARAPPAV imports
from arappav.errors.taxonomy_math import MathErrorType
from arappav.errors.schema_math import (
    MathInjectedError,
    MathPerturberOutput,
    MathVerifierClaim,
    MathVerifierOutput,
    validate_math_perturber_output,
    validate_math_verifier_output,
)
from arappav.data.ingest_math import (
    load_math_dataset,
    build_math_splits,
    MATH_TOPICS,
)
from arappav.models.perturber import (
    PerturberModel,
    build_math_perturber_prompt,
)
from arappav.models.verifier import (
    VerifierModel,
    build_math_verifier_prompt,
)
from arappav.reward.reward_fns import compute_rewards, RewardOutput
from arappav.training.selfplay_loop import SelfPlayLoop

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

print(f"Math error types available: {len(list(MathErrorType))}")
print(f"Math topics available: {MATH_TOPICS}")

## 2. Download and Explore the Hendrycks MATH Dataset

In [ ]:
# Download a small subset for exploration
print("Downloading Hendrycks MATH dataset (algebra subset)...")
explore_ds = load_math_dataset(
    topics=["algebra", "counting_and_probability"],
    split="train",
    max_examples_per_topic=100,
)
print(f"Loaded {len(explore_ds)} examples")
print(f"Columns: {explore_ds.column_names}")

In [ ]:
# Show a few examples
for i in range(3):
    row = explore_ds[i]
    print(f"\n{'='*60}")
    print(f"Example {i+1} — Topic: {row['topic']}, Level: {row['level']}")
    print(f"{'='*60}")
    print(f"\nProblem:\n{row['problem'][:500]}...")
    print(f"\nSolution:\n{row['solution'][:500]}...")

In [ ]:
# Dataset statistics
print("\n--- Dataset Statistics ---")
print(f"Total examples: {len(explore_ds)}")

# By topic
topic_counts = Counter(explore_ds["topic"])
print(f"\nTopics:")
for topic, count in topic_counts.most_common():
    print(f"  {topic}: {count}")

# By level
level_counts = Counter(explore_ds["level"])
print(f"\nLevels:")
for level, count in sorted(level_counts.items()):
    print(f"  {level}: {count}")

# Solution length distribution
sol_lens = [len(s.split()) for s in explore_ds["solution"]]
print(f"\nSolution length (words): min={min(sol_lens)}, max={max(sol_lens)}, mean={np.mean(sol_lens):.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
topics, counts = zip(*topic_counts.most_common())
ax.bar(topics, counts)
ax.set_title("Examples by Topic")
ax.tick_params(axis="x", rotation=45)

ax = axes[1]
ax.hist(sol_lens, bins=30, edgecolor="black")
ax.set_title("Solution Length Distribution (words)")
ax.set_xlabel("Words")

plt.tight_layout()
plt.show()

## 3. Build Train/Val/Test Splits

In [ ]:
# Build full splits across all topics
# For quick experimentation, limit examples per topic
QUICK_RUN = True  # Set to False for full-scale training

if QUICK_RUN:
    max_train, max_val, max_test = 200, 50, 50
else:
    max_train, max_val, max_test = None, 200, 200  # None = all

print("Building math splits...")
math_splits = build_math_splits(
    topics=MATH_TOPICS,
    max_train_per_topic=max_train,
    max_val_per_topic=max_val,
    max_test_per_topic=max_test,
    seed=42,
)

for split_name, ds in math_splits.items():
    print(f"  {split_name}: {len(ds)} examples")

# Save for later use
math_splits.save_to_disk("../data/processed/math_splits")
print("\nSplits saved to ../data/processed/math_splits")

## 4. Error Taxonomy for Math Mode

The math error types are derived from Rittle-Johnson et al. (2025),
"Detecting Math Misconceptions: An AI Benchmark Dataset."

In [ ]:
print(f"{len(list(MathErrorType))} math error types:\n")
for err_type in MathErrorType:
    desc = MathErrorType.descriptions().get(err_type.value, "")
    print(f"  {err_type.value:35s} — {desc}")

# Show topic groups
print(f"\nError types grouped by topic:\n")
for topic, errors in MathErrorType.topic_groups().items():
    print(f"  {topic:15s}: {', '.join(e.value for e in errors)}")

## 5. Inspect Prompt Templates

In [ ]:
# Show what the Perturber and Verifier actually see
sample = math_splits["train"][0]
k = 2

print("=" * 60)
print("PERTURBER PROMPT (math mode):")
print("=" * 60)
p_prompt = build_math_perturber_prompt(sample["problem"], sample["solution"], k)
print(p_prompt[:1500])
print("...\n")

print("=" * 60)
print("VERIFIER PROMPT (math mode):")
print("=" * 60)
v_prompt = build_math_verifier_prompt(sample["problem"], sample["solution"])
print(v_prompt[:1500])
print("...")

## 6. Run a Single Episode (Sanity Check)

This runs one complete P→V→Reward cycle to verify the pipeline works
end-to-end before committing to full training.

In [ ]:
# Load small models for testing (adjust paths as needed)
PERTURBER_MODEL = "Qwen/Qwen2.5-3B-Instruct"
VERIFIER_MODEL = "Qwen/Qwen2.5-7B-Instruct"

print(f"Perturber: {PERTURBER_MODEL}")
print(f"Verifier:  {VERIFIER_MODEL}")

# Sample a problem
row = math_splits["train"][random.randint(0, len(math_splits["train"]) - 1)]
problem = row["problem"]
solution = row["solution"]
k = random.randint(1, 3)

print(f"\nTopic: {row['topic']}, Level: {row['level']}, k={k}")
print(f"\n--- Problem ---\n{problem[:400]}...")
print(f"\n--- Correct Solution ---\n{solution[:400]}...")

In [ ]:
# Initialize Perturber and generate
print("Initializing Perturber...")
perturber = PerturberModel(
    model_name_or_path=PERTURBER_MODEL,
    mode="math",
    use_vllm=False,
)

print(f"Generating perturbation with k={k}...")
p_out, p_err = perturber.generate(problem, k, solution=solution)

if p_out is None:
    print(f"❌ Perturber failed: {p_err}")
else:
    print(f"✅ Perturber generated {len(p_out.errors)} errors:")
    for err in p_out.errors:
        print(f"\n  [{err.error_type.value}] {err.error_id} @ step {err.step_index}")
        print(f"    Original:  {err.original_text[:150]}")
        print(f"    Injected:  {err.injected_text[:150]}")
        print(f"    Rationale: {err.rationale[:200]}")
    print(f"\n--- Perturbed Solution (first 500 chars) ---")
    print(p_out.perturbed_solution[:500])

In [ ]:
# Initialize Verifier and run
if p_out is not None:
    print("Initializing Verifier...")
    verifier = VerifierModel(
        model_name_or_path=VERIFIER_MODEL,
        mode="math",
        use_vllm=False,
    )

    print("Generating verifier response...")
    results = verifier.generate(
        p_out.perturbed_solution, problem=problem, n_completions=1,
    )
    raw_text, v_out, v_err = results[0]

    if v_out is None:
        print(f"❌ Verifier failed: {v_err}")
    else:
        print(f"✅ Verifier made {len(v_out.claims)} claims:")
        print(f"   Raw output (first 300 chars):\n   {raw_text[:300]}...")
        for i, claim in enumerate(v_out.claims):
            print(f"\n  Claim {i+1}:")
            step_str = f" @ step {claim.step_index}" if claim.step_index is not None else ""
            print(f"    Quoted{step_str}: {claim.quoted_text[:150]}")
            print(f"    Explanation: {claim.explanation[:200]}")

In [ ]:
# Compute rewards
if p_out is not None and v_out is not None:
    reward_out = compute_rewards(
        ground_truth=p_out.errors,
        verifier_claims=v_out.claims,
        perturbed_text=p_out.perturbed_solution,
        k=k,
    )

    print("=" * 50)
    print("REWARD COMPUTATION")
    print("=" * 50)
    print(f"  Verifier Recall:     {reward_out.verifier_recall:.4f}")
    print(f"  Verifier Precision:  {reward_out.verifier_precision:.4f}")
    print(f"  Verifier F1:         {reward_out.verifier_f_beta:.4f}")
    print(f"  Verifier Reward:     {reward_out.verifier_reward:.4f}")
    print(f"  Perturber Reward:    {reward_out.perturber_reward:.4f}")
    print(f"  Format Penalty:      {reward_out.format_penalty_applied}")
    print(f"  Spam Penalty:        {reward_out.spam_penalty:.4f}")

## 7. Run Self-Play Training (Math Mode)

This runs the full alternating GRPO+DPO training loop on the math dataset.
Configure the settings below before running.

In [ ]:
# Training configuration
TRAINING_CONFIG = {
    # Self-play
    "num_rounds": 10,
    "episodes_per_round": 64,
    "verifier_episodes_per_round": 32,
    "group_size_G": 4,
    "pairs_per_update": 32,
    "freeze": null,                 # null (train both), "perturber" (train V only), "verifier" (train P only)
    "checkpoint_frequency": 2,
    "eval_frequency": 2,
    "seed": 42,

    # Models
    "perturber_model": "Qwen/Qwen2.5-3B-Instruct",
    "verifier_model": "Qwen/Qwen2.5-7B-Instruct",

    # Math-specific
    "topics": ["algebra", "counting_and_probability", "number_theory", "prealgebra"],
    "max_train_per_topic": 200,   # limit for quick iteration
    "max_val_per_topic": 50,
    "max_test_per_topic": 50,
    "k_range": [1, 3],

    # Training hyperparams
    "grpo_lr": 5e-6,
    "dpo_lr": 5e-6,
    "grpo_beta": 0.04,
    "dpo_beta": 0.1,

    # Output
    "output_dir": "./outputs_math",
    "rollout_dir": "./data/rollouts_math",
}

print("Training configuration:")
print(json.dumps(TRAINING_CONFIG, indent=2))

In [ ]:
# Build Hydra-compatible config
from omegaconf import OmegaConf

cfg = OmegaConf.create({
    "mode": "math",
    "self_play": {
        "num_rounds": TRAINING_CONFIG["num_rounds"],
        "episodes_per_round": TRAINING_CONFIG["episodes_per_round"],
        "verifier_episodes_per_round": TRAINING_CONFIG["verifier_episodes_per_round"],
        "group_size_G": TRAINING_CONFIG["group_size_G"],
        "pairs_per_update": TRAINING_CONFIG["pairs_per_update"],
        "freeze": TRAINING_CONFIG["freeze"],   # null (train both), "perturber", or "verifier"
        "checkpoint_frequency": TRAINING_CONFIG["checkpoint_frequency"],
        "eval_frequency": TRAINING_CONFIG["eval_frequency"],
        "seed": TRAINING_CONFIG["seed"],
    },
    "perturber": {
        "model_name_or_path": TRAINING_CONFIG["perturber_model"],
        "use_peft": True,
        "lora": {"r": 64, "alpha": 128, "dropout": 0.05,
                 "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"]},
        "generation": {"max_new_tokens": 2048, "temperature": 0.8, "top_p": 0.95, "do_sample": True},
    },
    "verifier": {
        "model_name_or_path": TRAINING_CONFIG["verifier_model"],
        "use_peft": True,
        "lora": {"r": 64, "alpha": 128, "dropout": 0.05,
                 "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"]},
        "generation": {"max_new_tokens": 1024, "temperature": 0.6, "top_p": 0.95, "do_sample": True},
    },
    "grpo": {
        "learning_rate": TRAINING_CONFIG["grpo_lr"],
        "per_device_batch_size": 1,
        "gradient_accumulation_steps": 4,
        "num_epochs": 1,
        "max_grad_norm": 1.0,
        "warmup_ratio": 0.1,
        "lr_scheduler_type": "cosine",
        "optim": "adamw_torch",
        "beta": TRAINING_CONFIG["grpo_beta"],
        "num_generations": TRAINING_CONFIG["group_size_G"],
        "temperature": 0.9,
        "max_prompt_length": 4096,
        "use_vllm_for_rollouts": False,
    },
    "dpo": {
        "learning_rate": TRAINING_CONFIG["dpo_lr"],
        "per_device_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "num_epochs": 1,
        "max_grad_norm": 1.0,
        "warmup_ratio": 0.1,
        "lr_scheduler_type": "cosine",
        "optim": "adamw_torch",
        "beta": TRAINING_CONFIG["dpo_beta"],
        "max_prompt_length": 4096,
        "max_length": 6144,
        "loss_type": "sigmoid",
    },
    "reward": {
        "k_range": TRAINING_CONFIG["k_range"],
        "format_penalty": -10.0,
        "span_overlap_threshold": 0.5,
        "precision_recall_beta": 1.0,
        "verifier_reward_formula": "f_beta",
        "perturber_reward_formula": "one_minus_recall",
        "use_semantic_match": False,
        "anti_duplicate": {"enabled": True, "similarity_threshold": 0.85, "penalty": -5.0},
        "anti_spam": {"enabled": True, "max_claims_ratio": 3.0, "penalty_per_excess": -0.5},
    },
    "corpus": {
        "processed_dir": "./data/processed",
    },
    "math": {
        "topics": TRAINING_CONFIG["topics"],
        "max_train_per_topic": TRAINING_CONFIG["max_train_per_topic"],
        "max_val_per_topic": TRAINING_CONFIG["max_val_per_topic"],
        "max_test_per_topic": TRAINING_CONFIG["max_test_per_topic"],
        "k_range": TRAINING_CONFIG["k_range"],
    },
    "vllm": {},
    "wandb": {"enabled": False},
    "output_dir": TRAINING_CONFIG["output_dir"],
    "rollout_dir": TRAINING_CONFIG["rollout_dir"],
})

print(f"Starting self-play training: {cfg.self_play.num_rounds} rounds, mode={cfg.mode}")
print(f"Freeze mode: {cfg.self_play.freeze}")  # null = train both
print(f"Topics: {cfg.math.topics}")
print(f"Training examples per topic: {cfg.math.max_train_per_topic}")

# Run the self-play loop
loop = SelfPlayLoop(cfg)
loop.run()

print("\n✅ Training complete!")

## 8. Evaluate and Visualize Results

In [ ]:
# Load rollouts from training
ROLLOUT_DIR = Path(TRAINING_CONFIG["rollout_dir"])

def load_rollouts(prefix: str, mode: str = "math") -> list[dict]:
    rollouts = []
    pattern = f"{prefix}_{mode}_*.jsonl"
    for path in sorted(ROLLOUT_DIR.glob(pattern)):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    rollouts.append(json.loads(line))
    return rollouts

p_rollouts = load_rollouts("perturber")
v_rollouts = load_rollouts("verifier")
print(f"Loaded {len(p_rollouts)} Perturber rollouts")
print(f"Loaded {len(v_rollouts)} Verifier rollouts")

In [ ]:
# Analyze error type distribution from Perturber rollouts
if p_rollouts:
    type_counts = Counter()
    k_values = []
    format_valid = 0

    for r in p_rollouts:
        if r.get("format_valid"):
            format_valid += 1
        if r.get("ground_truth"):
            k_values.append(len(r["ground_truth"]))
            for err in r["ground_truth"]:
                type_counts[err.get("error_type", "unknown")] += 1

    print(f"Format compliance: {format_valid}/{len(p_rollouts)} ({format_valid/len(p_rollouts):.1%})")
    print(f"Avg k: {np.mean(k_values):.1f} (range: [{min(k_values)}, {max(k_values)}])")

    # Plot error type distribution
    if type_counts:
        types, counts_vals = zip(*type_counts.most_common(15))
        fig, ax = plt.subplots(figsize=(12, 5))
        bars = ax.bar(range(len(types)), counts_vals)
        ax.set_xticks(range(len(types)))
        ax.set_xticklabels(types, rotation=45, ha="right")
        ax.set_title("Math Error Types Injected by Perturber")
        ax.set_ylabel("Count")
        for bar, count in zip(bars, counts_vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    str(count), ha="center", va="bottom", fontsize=8)
        plt.tight_layout()
        plt.show()
else:
    print("No rollouts found. Run the training loop first.")

In [ ]:
# Analyze Verifier performance from rollouts
if v_rollouts:
    n_claims_list = []

    for r in v_rollouts:
        k = r.get("k", 0)
        n_gt = len(r.get("ground_truth", []))
        for resp in r.get("responses", []):
            # responses are dicts with "raw_text", "parsed", "parse_error"
            parsed = resp.get("parsed") if isinstance(resp, dict) else (resp[1] if len(resp) > 1 else None)
            if parsed and "claims" in parsed:
                n_claims_list.append(len(parsed["claims"]))

    if n_claims_list:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        ax = axes[0]
        ax.hist(n_claims_list, bins=range(0, max(n_claims_list) + 2), edgecolor="black", align="left")
        ax.set_title("Verifier Claims per Episode")
        ax.set_xlabel("Number of Claims")
        ax.set_ylabel("Frequency")

        ax = axes[1]
        claim_counter = Counter(n_claims_list)
        xs, ys = zip(*sorted(claim_counter.items()))
        ax.bar(xs, ys, edgecolor="black")
        ax.set_title("Verifier Claims Distribution")
        ax.set_xlabel("Number of Claims")

        plt.tight_layout()
        plt.show()

        print(f"Claims stats: mean={np.mean(n_claims_list):.1f}, median={np.median(n_claims_list):.0f}, max={max(n_claims_list)}")
else:
    print("No Verifier rollouts found.")

## 9. Load Checkpoint and Run Manual Evaluation

Test the trained models on held-out problems.

In [ ]:
# Load a checkpoint and test on new problems
CHECKPOINT_DIR = Path(TRAINING_CONFIG["output_dir"])

# Find latest checkpoint
p_checkpoints = sorted(CHECKPOINT_DIR.glob("perturber_math_round*"))
v_checkpoints = sorted(CHECKPOINT_DIR.glob("verifier_math_round*"))

if p_checkpoints and v_checkpoints:
    p_ckpt = p_checkpoints[-1]
    v_ckpt = v_checkpoints[-1]
    print(f"Latest Perturber checkpoint: {p_ckpt}")
    print(f"Latest Verifier checkpoint:  {v_ckpt}")

    # Load models from checkpoints
    eval_perturber = PerturberModel(
        model_name_or_path=str(p_ckpt),
        mode="math",
        use_vllm=False,
    )
    eval_verifier = VerifierModel(
        model_name_or_path=str(v_ckpt),
        mode="math",
        use_vllm=False,
    )

    # Run on test set
    from arappav.eval.evaluate import run_evaluation
    from arappav.eval.metrics import format_metrics_report

    metrics = run_evaluation(
        perturber_model=eval_perturber,
        verifier_model=eval_verifier,
        eval_dataset=math_splits["test"],
        num_episodes=20,
        qualitative_samples=3,
        output_dir=str(CHECKPOINT_DIR / "eval_math"),
    )

    print(format_metrics_report(metrics))
else:
    print("No checkpoints found. Complete training first.")

## 10. Summary

### What this notebook does
1. Downloads the Hendrycks MATH dataset (7 topics, 5 difficulty levels)
2. Builds train/val/test splits
3. Shows the 26 math-specific error types derived from misconception research
4. Runs a single P→V→Reward episode for sanity checking
5. Runs the full self-play training loop on math problems
6. Evaluates and visualizes the results

### How the adversarial game works in math mode
- **Perturber** sees: problem + correct solution → injects k misconception errors
- **Verifier** sees: problem + perturbed solution → finds and explains errors
- **Perturber wins** when Verifier misses errors; **Verifier wins** when it catches them
- Both improve over rounds through GRPO (Perturber) and DPO (Verifier)

### Key differences from paper mode
| Aspect | Paper Mode | Math Mode |
|--------|-----------|----------|
| Input | Paper section text | Problem + solution pair |
| Error types | 6 (numerical, citation, logical, ...) | 26 (misconception-based) |
| Dataset | Local text files | Hendrycks MATH (HF Hub) |
| Perturber task | Inject errors into text | Inject errors into solution steps |
| Verifier task | Find errors in text | Find errors in solution |